# Hateful Memes V3 — Full Training & Analysis Pipeline

Runs the complete resubmission pipeline on a **T4 GPU** runtime (Colab or Kaggle):
1. Train **Dynamic V3** (adaptive gating) and **Static V3** (α=0.5 baseline)
2. **Significance testing** — paired bootstrap CIs + McNemar's test (Reviewer 3)
3. **Inference cost profile** — params / latency / FLOPs (Reviewers 2 & 3)
4. Optional **ensemble** of the two models

Checkpoints are written to Google Drive every `CHECKPOINT_EVERY` epochs (rolling `*_last.pt`),
so a disconnect costs at most that many epochs — rerun the training cell with `RESUME=true`.

**Kaggle instead of Colab:** upload the bundle as a Kaggle Dataset, skip the Drive cells, and point
`DATA_DIR` at `/kaggle/input/<dataset-name>/Data` and `CHECKPOINT_DIR` at `/kaggle/working/checkpoints`.

## 1. GPU check

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda :', torch.cuda.is_available() and torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), 'Switch runtime to GPU (Runtime → Change runtime type → T4)'

## 2. Dependencies (pinned)

`torch` is deliberately **not** reinstalled — the Colab/Kaggle image ships a CUDA-matched build
(developed against torch 2.10 / transformers 5.3; any torch ≥ 2.1 works).

In [ ]:
%pip -q install 'transformers==5.3.0' 'optuna==4.*' scikit-learn scipy pandas pyarrow pillow tqdm
import transformers, sklearn, scipy, optuna
print('transformers', transformers.__version__, '| sklearn', sklearn.__version__,
      '| scipy', scipy.__version__, '| optuna', optuna.__version__)

## 3. Mount Google Drive

The **code** is cloned fresh from GitHub each run; Drive only provides the **data**.
Upload `hateful_memes_colab_bundle.zip` (built by `python colab/make_bundle.py`)
to `MyDrive/` once — you never need to re-upload it after code changes.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Get code (GitHub) + data (Drive bundle)

In [ ]:
from pathlib import Path
import shutil, subprocess, zipfile

REPO_URL    = 'https://github.com/lucenity0/A-Unified-Adaptive-Framework-for-Multimodal-Data-Fusion-with-Dynamic-Modality-Reweighting.git'
BUNDLE_PATH = Path('/content/drive/MyDrive/hateful_memes_colab_bundle.zip')
PROJECT_DIR = Path('/content/hateful_memes_v3')

# fresh clone of the latest code
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)

# data comes from the bundle already in Drive (parquet files only)
assert BUNDLE_PATH.exists(), f'Bundle not found: {BUNDLE_PATH} — upload it to Drive once'
with zipfile.ZipFile(BUNDLE_PATH) as zf:
    members = [m for m in zf.namelist() if m.startswith('Data/')]
    zf.extractall(PROJECT_DIR, members=members)

!ls {PROJECT_DIR}/src
!ls -lh {PROJECT_DIR}/Data

## 5. Configure the run

Everything below is consumed by `src/config.py` — no script edits needed.
For a quick pipeline sanity check first, uncomment `MAX_SAMPLES`.

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path('/content/hateful_memes_v3')
CKPT_DIR    = Path('/content/drive/MyDrive/hateful_memes_checkpoints_v3')
RESULTS_DIR = Path('/content/drive/MyDrive/hateful_memes_results_v3')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

os.environ['DATA_DIR']         = str(PROJECT_DIR / 'Data')
os.environ['CHECKPOINT_DIR']   = str(CKPT_DIR)
os.environ['RESULTS_DIR']      = str(RESULTS_DIR)
os.environ['BATCH_SIZE']       = '8'      # if CUDA OOMs: 4 (+ GRAD_ACCUM=8)
os.environ['GRAD_ACCUM']       = '4'
os.environ['NUM_EPOCHS']       = '20'
os.environ['PATIENCE']         = '4'
os.environ['CHECKPOINT_EVERY'] = '2'
os.environ['CURRICULUM']       = 'true'
os.environ['RESUME']           = 'false'  # set 'true' when re-running after a disconnect
# os.environ['MAX_SAMPLES']    = '128'    # uncomment for a quick smoke run

# Loss-weight sweep knobs (defaults come from the repo config):
# os.environ['ENTROPY_WEIGHT']   = '0.002'  # 0.05 collapsed alpha to a constant 0.5
# os.environ['DIVERSITY_WEIGHT'] = '0.1'    # raise if alpha std still shrinks
# os.environ['RDROP_WEIGHT']     = '0.2'    # lower if gate stays inexpressive
# os.environ['AUX_WEIGHT']       = '0.1'

for k in ['DATA_DIR','CHECKPOINT_DIR','RESULTS_DIR','BATCH_SIZE','GRAD_ACCUM',
          'NUM_EPOCHS','PATIENCE','CHECKPOINT_EVERY','CURRICULUM','RESUME']:
    print(f'{k} = {os.environ[k]}')

## 6. Train — Dynamic V3

After a disconnect: rerun cell 5 with `RESUME='true'`, then this cell.

In [ ]:
%cd /content/hateful_memes_v3/src
!python train_dynamic_v3.py

## 7. Train — Static V3 baseline

In [ ]:
%cd /content/hateful_memes_v3/src
!python train_static_v3.py

## 8. Significance testing (Reviewer 3)

Paired bootstrap (1,000 resamples, seed 42): per-model 95% CIs for Table 3 plus Δ CIs and
p-values; McNemar's test on paired decisions at each model's validation-selected threshold.
Writes `significance_report.txt` and `paired_val_predictions.csv` to `RESULTS_DIR`.

In [ ]:
%cd /content/hateful_memes_v3/src
!python significance.py

## 9. Inference cost profile (Reviewers 2 & 3)

Parameter deltas over vanilla CLIP, per-sample latency (bs=1 and bs=8) **measured on the T4**,
FLOPs estimate. Writes `cost_report.txt` to `RESULTS_DIR`.

In [ ]:
%cd /content/hateful_memes_v3/src
!python cost_profile.py

## 10. Optional: ensemble (dynamic + static logit blend)

In [ ]:
%cd /content/hateful_memes_v3/src
import os
ck = os.environ['CHECKPOINT_DIR']
!DYNAMIC_CKPT={ck}/best_model_dynamic_v3.pt STATIC_CKPT={ck}/best_model_static_v3.pt python - <<'PY'
import os
from ensemble import run_ensemble, CONFIG
CONFIG['val_parquet']  = os.path.join(os.environ.get('DATA_DIR', '../Data'),
                                      'validation-00000-of-00001-1508d9e5032c2c1f.parquet')
CONFIG['dynamic_ckpt'] = os.environ['DYNAMIC_CKPT']
CONFIG['static_ckpt']  = os.environ['STATIC_CKPT']
CONFIG['output_dir']   = os.environ.get('RESULTS_DIR', '../results')
run_ensemble(CONFIG)
PY

## Outputs to pull into the paper

| File (in `RESULTS_DIR` on Drive) | Paper destination |
|---|---|
| training cell logs (best AUROC/acc, best epoch, alpha stats) | Table 3, §IV.E |
| `significance_report.txt` | Table 3 CI columns + significance sentence in §IV.D |
| `paired_val_predictions.csv` | reproducibility archive |
| `cost_report.txt` | new computational-cost subsection |
| `best_model_*_v3.pt` in `CHECKPOINT_DIR` | keep for CrisisMMD transfer runs |